# Coupling-Phase Spectroscopy: instrument calibration

This notebook verifies the mathematical instrument before any model checkpoint is downloaded. It is deliberately verbose: every stage announces what it is testing, why the test matters, and where the evidence is written.

## Learning objectives

By the end of the run, you should be able to distinguish:

1. a phase-family construction from an arbitrary perturbation;
2. eigenvalue continuation from independent eigenvalue sorting;
3. asymptotic spectral radius from finite-horizon transient gain;
4. a software smoke test from empirical evidence about Pythia training.

**Evidence boundary.** Passing this notebook validates the implementation and synthetic fixtures. It does not validate any claim about a language model or optimizer trajectory.

In [ ]:
import os, pathlib, subprocess, sys, time
from IPython.display import Markdown, display

REPO_URL = os.environ.get("CPS_REPO_URL", "https://github.com/fyremael/CPS.git")
GIT_REF = os.environ.get("CPS_GIT_REF", "main")
repo = pathlib.Path("/content/CPS")

print("[BOOT] Preparing the CPS repository", flush=True)
print(f"[BOOT] source={REPO_URL}", flush=True)
print(f"[BOOT] ref={GIT_REF}", flush=True)
if not repo.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(repo)], check=True)
else:
    subprocess.run(["git", "-C", str(repo), "fetch", "origin", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "checkout", GIT_REF], check=True)
    subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "origin", GIT_REF], check=True)
os.chdir(repo)
print("[BOOT] Installing CPS with Pythia and notebook dependencies", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "-e", ".[pythia,notebooks]"
], check=True)
print(f"[BOOT] Ready: {repo}", flush=True)

# Editable installs write a .pth file, but the running Colab kernel does not
# automatically reprocess newly-created .pth files. Put the source tree on
# sys.path explicitly so the very next cell can import CPS without a restart.
import importlib
src_dir = repo / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))
importlib.invalidate_caches()
import cps
print(f"[BOOT] CPS import verified from {cps.__file__}", flush=True)


In [ ]:
from cps.notebook import show_environment
runtime = show_environment()

## Stage 1 — test the structural contracts

The selected tests exercise magnitude-preserving perturbations, eigenvalue tracking, projected operators, and the new JVP fallback policy. Verbose test names are shown so failures are locally interpretable.

In [ ]:
import subprocess, sys
command = [
    sys.executable, "-m", "pytest", "-vv",
    "tests/test_perturbations.py",
    "tests/test_spectra.py",
    "tests/pythia/test_reduced_operator.py",
    "tests/pythia/test_jvp.py",
]
print("[TEST]", " ".join(command), flush=True)
subprocess.run(command, check=True)

## Stage 2 — generate a synthetic optimizer-like example

The synthetic experiment supplies a controlled matrix for which spectral motion and transient amplification can be inspected without checkpoint, tokenizer, or fused-kernel confounders.

In [ ]:
import subprocess, sys
print("[SYNTHETIC] Generating the quadratic-system evidence packet", flush=True)
subprocess.run([sys.executable, "experiments/synthetic_quadratics.py"], check=True)
print("[SYNTHETIC] Complete", flush=True)

In [ ]:
from cps.notebook import export_artifacts
archive = export_artifacts()
print(f"Artifact archive ready for colab-cli download: {archive}", flush=True)